# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivanilokh/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **One Row Meaning**: One unique row represents the daily organic search performance metrics (impressions, clicks, CTR) for a unique pseudonymized content item (URL) belonging to a specific client on a single calendar date.
* **Target Window**: We are using the mid-panel month of March 2026 (`month=2026-03`) to build our features and establish the baseline training dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Primary Tables**: `fact_content_daily_performance`, joined with `dim_content` and `dim_clients`.
* **Target Label**: A binary proxy for high content performance — whether a page achieved a Click-Through Rate (CTR) above the median for its client slot on that day, with non-zero impressions.
* **Deliberately Excluded**: We explicitly drop all rows where `impressions = 0` because CTR becomes mathematically undefined, and we exclude the first 3 days of newly created content to avoid indexing volatility noise.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

import duckdb

# 1. Hugging Face se connect karne ke liye secret token setup
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN 'hf_your_read_token')")

# Base Path define kar rahe hain mid-panel month (March 2026) ke liye
rel_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("--- Query 1: Grain Check ---")
# Yeh check karega ki (report_date, client_hash, content_hash) milakar data unique hai ya nahi
q1 = f"""
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT CONCAT(report_date, '-', client_hash, '-', content_hash)) as unique_grain_count
FROM read_parquet('{rel_path}')
WHERE impressions > 0;
"""
print(con.sql(q1).to_df())

print("\n--- Query 2: Row Count & Date Range ---")
# Yeh check karega ki data sach mein March 1 se March 31, 2026 tak ka hai aur total kitne rows hain
q2 = f"""
SELECT
    MIN(report_date) as slice_start,
    MAX(report_date) as slice_end,
    COUNT(*) as slice_total_rows
FROM read_parquet('{rel_path}')
WHERE impressions > 0;
"""
print(con.sql(q2).to_df())

print("\n--- Query 3: Availability Check (IS TRUE) ---")
# IS TRUE check jo dikhayega hamare safety filters ke baad kitna data bacha
q3 = f"""
SELECT
    COUNT(*) as rows_entering,
    SUM(CASE WHEN (impressions > 0 AND clicks >= 0) IS TRUE THEN 1 ELSE 0 END) as rows_surviving
FROM read_parquet('{rel_path}');
"""
print(con.sql(q3).to_df())

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.